In [9]:
from typing import cast
from datasets import DatasetDict, load_dataset


In [28]:
def extract_gsm8k_final_answer(text: str) -> str:
    """Extract the final numeric/string answer from a GSM8K solution field.

    GSM8K format typically places the final answer on a line starting with
    '####'. We take the substring following '####' on the last such line.
    """
    import re
    lines = text.splitlines()
    for line in reversed(lines):
        s = line.strip()
        if s.startswith("####"):
            content = s[4:].strip()
            if content.startswith(":"):
                content = content[1:].strip()
            content = content.replace(",", "").strip()
            return content
    matches = re.findall(r"####\s*(.+)", text)
    if matches:
        return matches[-1].strip()
    raise ValueError("No GSM8K final answer found")

def extract_token_string(text: str, token: str = "think") -> str:
    """Extract everything between <token> and </token> tags.
    
    Returns the content between the first occurrence of <token> and </token>.
    If no match is found, raises ValueError.
    """
    import re
    pattern = rf'<{token}>(.*?)</{token}>'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1)
    raise ValueError("No <token>...</token> tags found")

In [29]:
test = """
<think>
This is a test thinking...
</think>
<answer>
This is a test answer.
</answer>
"""

In [31]:
print(extract_token_string(test, "think"))
print(extract_token_string(test, "answer"))


This is a test thinking...


This is a test answer.



In [22]:
ds = cast(DatasetDict, load_dataset("openai/gsm8k", name="main"))
train_dataset = ds["train"]

In [23]:
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [ ]:
extract_gsm8k_final_answer(train_dataset[0]["answer"])

72